In [12]:
!pip install -q transformers sentencepiece accelerate

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

from transformers.modeling_outputs import BaseModelOutput

from tqdm.auto import tqdm

In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [15]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

for p in model.parameters():
    p.requires_grad = False

print("Model loaded.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded.


In [16]:
class PerceiverCompressor(nn.Module):
    def __init__(
        self,
        hidden_size=768,
        num_latents=128,
        num_heads=12,
    ):
        super().__init__()

        self.latents = nn.Parameter(
            torch.randn(num_latents, hidden_size)
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states):

        batch_size = hidden_states.size(0)

        latents = self.latents.unsqueeze(0).expand(
            batch_size,
            -1,
            -1,
        )

        compressed, _ = self.cross_attn(
            query=latents,
            key=hidden_states,
            value=hidden_states,
        )

        compressed = self.norm(compressed)

        return compressed

In [17]:
compressor = PerceiverCompressor().to(device)

compressor.load_state_dict(
    torch.load(
        "compressor_256_to_128.pt",
        map_location=device,
    )
)

print("Checkpoint loaded.")

Checkpoint loaded.


In [18]:
TEMPERATURE = 2.0
MSE_WEIGHT = 0.1

def distillation_kl(
    student_logits,
    teacher_logits,
    temperature=2.0,
):

    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=-1,
    )

    teacher_probs = F.softmax(
        teacher_logits / temperature,
        dim=-1,
    )

    loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    )

    loss *= temperature ** 2

    return loss

In [19]:
def training_step(
    input_ids,
    attention_mask,
):

    decoder_input_ids = model._shift_right(
        input_ids
    )

    with torch.no_grad():

        teacher_encoder = model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        teacher_hidden = (
            teacher_encoder.last_hidden_state
        )

        teacher_outputs = model(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=teacher_hidden
            ),
            decoder_input_ids=decoder_input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        teacher_logits = teacher_outputs.logits

        teacher_decoder_hidden = (
            teacher_outputs.decoder_hidden_states[-1]
        )

    compressed = compressor(
        teacher_hidden
    )

    student_outputs = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=compressed
        ),
        decoder_input_ids=decoder_input_ids,
        output_hidden_states=True,
        return_dict=True,
    )

    student_logits = student_outputs.logits

    student_decoder_hidden = (
        student_outputs.decoder_hidden_states[-1]
    )

    kl_loss = distillation_kl(
        student_logits,
        teacher_logits,
        TEMPERATURE,
    )

    mse_loss = F.mse_loss(
        student_decoder_hidden,
        teacher_decoder_hidden,
    )

    total_loss = (
        kl_loss
        + MSE_WEIGHT * mse_loss
    )

    return (
        total_loss,
        kl_loss.detach(),
        mse_loss.detach(),
    )

In [20]:
compressor.train()

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1e-5,
    weight_decay=1e-4,
)

In [21]:
dataset = load_dataset(
      "ag_news"
)

train_texts = []

for x in dataset["train"]:
    txt = x["text"]

    if len(txt) > 20:
        train_texts.append(txt)

print(
    len(train_texts)
)

NameError: name 'load_dataset' is not defined

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        tokenizer,
        max_length=256,
    ):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].squeeze(0)

        attention_mask = enc[
            "attention_mask"
        ].squeeze(0)

        labels = input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [ ]:
train_dataset = TextDataset(
    train_texts,
    tokenizer,
    max_length=256,
)

print(len(train_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
)

In [ ]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

loss, kl, mse = training_step(
    input_ids,
    attention_mask,
)

print("Loss:", loss.item())
print("KL:", kl.item())
print("MSE:", mse.item())

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):

    compressor.train()

    running_loss = 0

    pbar = tqdm(train_loader)

    for batch in pbar:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        loss, kl, mse = training_step(
            input_ids,
            attention_mask,

        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            compressor.parameters(),
            1.0,
        )

        optimizer.step()

        running_loss += loss.item()

        pbar.set_description(
            f"loss={loss.item():.4f} "
            f"kl={kl.item():.4f} "
            f"mse={mse.item():.4f}"
        )

    print(
        f"Epoch {epoch+1}: "
        f"{running_loss/len(train_loader):.4f}"
    )

In [ ]:
torch.save(
    compressor.state_dict(),
    "compressor_hybrid_kl_mse.pt",
)

print("Saved.")

In [ ]:
def compare_generation(text):

    compressor.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():

        encoder_hidden = model.encoder(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
        ).last_hidden_state

        compressed = compressor(
            encoder_hidden
        )

        teacher = model.generate(
            **inputs,
            max_new_tokens=40,
        )

        student = model.generate(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=compressed
            ),
            attention_mask=torch.ones(
                compressed.shape[:2],
                dtype=torch.long,
                device=device,
            ),
            max_new_tokens=40,
        )

    print(
        "Teacher:",
        tokenizer.decode(
            teacher[0],
            skip_special_tokens=True,
        ),
    )

    print()

    print(
        "Student:",
        tokenizer.decode(
            student[0],
            skip_special_tokens=True,
        ),
    )